In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Running this notebook on: ", device)

import spateo as st
print("Last run with spateo version:", st.__version__)

# Other imports
import warnings, string
import anndata as ad
import dynamo as dyn
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Uncomment the following if running on the server
import pyvista as pv
pv.start_xvfb()

sns.set_theme(context="paper", style="ticks", font_scale=1)
warnings.filterwarnings('ignore')
# %load_ext autoreload

Running this notebook on:  cpu


2025-06-14 12:44:02.845716: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/root/miniforge3/envs/spateo/lib/python3.8/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


fastpd is not installed. If you need mesh correction, please compile the fastpd library.
Last run with spateo version: 0.0.0


In [2]:
adata = sc.read_h5ad('/data/work/05.cluster/FuseMap/0106/Hippocampus_latent_embeddings_all_single_pretrain/dmt_leiden_20250108_1.h5ad')
adata

AnnData object with n_obs × n_vars = 1112773 × 33326
    obs: 'dnbCount', 'area', 'orig.ident', 'x', 'y', 'region', 'n_counts', 'region_h2', 'Tangram_1119_celltype', 'Tangram_1119_celltype_main_frac', 'region_hip', 'slice_code', 'sub_region', 'dmt_leiden', 'dmt_leiden_merge', 'dmt_leiden_anno'
    uns: 'dmt_leiden_colors', 'dmt_nn', 'leiden', 'slice_code_colors'
    obsm: 'X_dmt', 'X_dmt_highdim', 'align_spatial_2d', 'align_spatial_3d', 'cell_border', 'latent_embeddings_all_single_pretrain', 'latent_embeddings_all_spatial_pretrain', 'spatial', 'spatial_division'
    obsp: 'dmt_nn_connectivities', 'dmt_nn_distances'

In [3]:
names = [
    # 'B03607C4E6_WT2024071214941.h5ad',
    
    '12_B03605F3G5_WT202403310048.h5ad',
    '13_B03612A1C3_WT202403310056.h5ad',
    '14_A03591A1C3_WT202403310045.h5ad',
    '16_A03592A4C6_WT202403310044.h5ad',
    '18_B03602C4D6_WT202405020031.h5ad',
    '20_B03606F3G5_WT202405020032.h5ad',
    '22_B03606C4E6_WT202403310050.h5ad',
    '23_B03609A4D6_WT202404150263.h5ad',
    '27_B03610C1E3_WT202403310051.h5ad',
    '31_B03619A1D3_WT202403310052.h5ad',
    '35_B03619E4G6_WT202403310053.h5ad',
    '39_A03589A1D4_WT202403310046.h5ad',
    '43_A03590E1G4_WT202403310064.h5ad',
    '47_A03593C1F3_WT202403310068.h5ad',
]
adata = adata[adata.obs['slice_code'].isin(names)]

In [6]:
adatas = []
for i in set(adata.obs['slice_code']):
    temp = adata[adata.obs['slice_code'] == i].copy()
    temp = temp[temp.obs.sample(frac=0.1).index].copy()
    if i == '47_A03593C1F3_WT202403310068.h5ad':
        temp.obsm['align_spatial_3d'] = np.array([temp.obsm['align_spatial_2d'][:, 0],
                                                  temp.obsm['align_spatial_2d'][:, 1], 
                                                  [1187.5 for i in range(len(temp))]]).T

    # sc.pp.normalize_total(temp)
    # sc.pp.log1p(temp)
    # sc.pp.scale(temp, zero_center=False, max_value=10)
    adatas.append(temp)
adata = ad.concat(adatas)

In [7]:
dic_dmt_leiden = {
   
    
    '2': 'ctx_sc_01',
    '13': 'ctx_sc_02',
    '19': 'ctx_sc_03',
    '20': 'ctx_sc_04',
    
    '4': 'ctx_sc_05',
    '21': 'ctx_sc_05',
    
    '8': 'ctx_sc_06',
    '10': 'ctx_sc_07',
    '22': 'ctx_sc_08',
    
    '9': 'ctx_sc_09',
    '11': 'ctx_sc_09',
    '16': 'ctx_sc_10',
    '18': 'ctx_sc_10',
    '26': 'ctx_sc_10',
    '31': 'ctx_sc_10',
    
    '12': 'ctx_sc_11',
    '15': 'ctx_sc_12',
    '25': 'ctx_sc_12',
    '29': 'ctx_sc_12',
    '17': 'ctx_sc_13',
    
    '24': 'z_delete',
    
    
    '5': 'ctx_sc_14',
    '23': 'ctx_sc_15',
    '27': 'ctx_sc_16',
    
    '30': 'ctx_sc_17',
    
     '0': 'hip_sc_18',
    '7': 'hip_sc_19',
    
    '1': 'hip_sc_20',
    '3': 'hip_sc_21',
    '6': 'hip_sc_21',
    '14': 'hip_sc_22',
    '28': 'hip_sc_23',
}
adata.obs['dmt_leiden_anno'] = [dic_dmt_leiden[i] for i in adata.obs['dmt_leiden']]
adata = adata[adata.obs['dmt_leiden_anno'] != 'z_delete']
colormap = {
 
  'ctx_sc_01' : '#374898',
  'ctx_sc_02' : '#6d85c7',
  'ctx_sc_03' : '#35c498',
  'ctx_sc_04' : '#9e2dc6',
  'ctx_sc_05' : '#2d7476',
  'ctx_sc_06' : '#cb0d6c',
  'ctx_sc_07' : '#20ea38',
  'ctx_sc_08' : '#0fabb6',
  'ctx_sc_09' : '#a59099',
  'ctx_sc_10' : '#2bea3a',
  'ctx_sc_11' : '#17b064',
  'ctx_sc_12' : '#52b8d5',
  'ctx_sc_13' : '#da2ef2',
  'ctx_sc_14' : '#6240f7',
  'ctx_sc_15' : '#c47233',
  'ctx_sc_16':'#a83b23',
  'ctx_sc_17':'#9994da',
  'hip_sc_18' : '#9b38e9',
  'hip_sc_19' : '#a89630',
  'hip_sc_20': '#5b798b',
  'hip_sc_21' : '#cb2505',
  'hip_sc_22' : '#62e7dd',
  'hip_sc_23' : '#245200',
}


In [8]:
adata.obsm['3d_align_spatial'] = adata.obsm['align_spatial_3d'][:, [0, 2, 1]].copy()
adata.obsm['3d_align_spatial'][:,2] = -adata.obsm['3d_align_spatial'][:,2]
adata.obsm['3d_align_spatial'] = adata.obsm['3d_align_spatial'] - adata.obsm['3d_align_spatial'].mean(axis = 0)

In [9]:
colormap.keys()

dict_keys(['ctx_sc_01', 'ctx_sc_02', 'ctx_sc_03', 'ctx_sc_04', 'ctx_sc_05', 'ctx_sc_06', 'ctx_sc_07', 'ctx_sc_08', 'ctx_sc_09', 'ctx_sc_10', 'ctx_sc_11', 'ctx_sc_12', 'ctx_sc_13', 'ctx_sc_14', 'ctx_sc_15', 'ctx_sc_16', 'ctx_sc_17', 'hip_sc_18', 'hip_sc_19', 'hip_sc_20', 'hip_sc_21', 'hip_sc_22', 'hip_sc_23'])

In [10]:
celltypes = ['ctx_sc_01', 'ctx_sc_02', 'ctx_sc_03', 'ctx_sc_04', 'ctx_sc_05', 'ctx_sc_06', 'ctx_sc_07', 'ctx_sc_08', 
             'ctx_sc_09', 'ctx_sc_10', 'ctx_sc_11', 'ctx_sc_12', 'ctx_sc_13', 'ctx_sc_14', 
             'ctx_sc_15', 'ctx_sc_16', 'ctx_sc_17', 'hip_sc_18', 'hip_sc_19', 'hip_sc_20', 'hip_sc_21', 'hip_sc_22', 'hip_sc_23']
for celltype in celltypes:
    colormap_c = colormap.copy()
    for c in colormap_c.keys():
        if c != celltype:
            colormap_c[c] = '#f5f5f520'
        else:
            colormap_c[c] = colormap_c[c]+'ff'
    pc, plot_cmap = st.tdr.construct_pc(adata=adata.copy(), spatial_key="3d_align_spatial", groupby="dmt_leiden_anno", key_added='region_anno', colormap = colormap_c)
    opacity = [0.1 if i != celltype else 0.8 for i in adata.obs['dmt_leiden_anno']]
    pc['region_anno_rgba'][:,3] = opacity
    st.pl.three_d_plot(
        model=pc,
        key='region_anno',
        # colormap=colormap_c,
        opacity=0.1,
        model_style="points",
        jupyter="html",
        background = '#EEEEEE',
        # cpo=[cpo]
        plotter_filename = f'3d_plot/celltype/{celltype}.html'
    )

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

...

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…